# Usine A100 : entrainer le classifieur (S1) de Prophet Studio, puis l'importer sur la RTX 5060

Chaine complete, cellule par cellule : **donnees Prophet -> (option) Bonsai 2 27B enseignant -> entrainement QLoRA ->
fusion -> GGUF (Q8_0, Q4_K_M) -> calibration -> telechargement**, puis import dans Prophet Studio
(Modeles > Importer un GGUF, role Classifieur, puis Calibrer).

Les donnees reproduisent exactement ce que Prophet demande au classifieur : pre-tour (`direct`, `clarify`, `intent`,
`language`, `needs_reasoning`, `risk`) sur `{request, workspace_files, recent_turns}`, pertinence des outils,
verification des reponses, commandes vocales, et garde-fou (avec **vos** exemples risques, voir la cellule Donnees).

Runtime : Execution > Modifier le type d'execution > **A100**. Tout ce qui compte est copie dans Google Drive au fil de
l'eau (session ephemere). Guide : `docs/05-COLAB-A100.md` de la branche clonee.

In [ ]:
# ---- parametres (modifier ici) -------------------------------------------------------------------------------------------
REPO = 'https://github.com/speed25200-cyber/LLM-and-Classifier.git'
BRANCH = 'claude/local-llm-high-performance-q4wk6f'   # branche de Prophet Studio (prophet_studio/, training/, colab/)
MODEL = 'Qwen/Qwen3.5-0.8B-Base'   # S1 sur CPU a cote de Bonsai 2 27B (RTX 5060 8 Go) : 0.8B conseille ; 2B = plus lent sur CPU
TRAIN_MODE = 'qlora'               # 'qlora' (base 4-bit + adaptateur, puis fusion), 'lora' (bf16) ou 'full' (tous les poids)
N_EXAMPLES = 20000                 # exemples generes par regles (uniques, au plus)
USE_TEACHER = False                # True : Bonsai 2 27B (PQ2_0 sur l'A100) etiquette l'entrainement (distributions + KL)
TEACHER_WORKERS = 4                # = slots du serveur Bonsai (profil a100 : 4)
GUARD_EXTRA = 'guard_risky.jsonl'  # optionnel, dans DRIVE : vos exemples de garde en plus des integres (format dans training/make_synthetic_prophet.py)
QUANTS = 'Q8_0,Q4_K_M'
LLAMA_TAG = 'prism-b10683-d8f26ee' # runtime de Prophet Studio : meme version pour la conversion et la calibration
NAME = 'jev-clone'                 # prefixe des fichiers GGUF (Studio en deduit l'id custom-s1-jev-clone-q8-0)
DRIVE = '/content/drive/MyDrive/prophet-s1'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv   # attendu : NVIDIA A100
import os, subprocess, time, json, shutil
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
# ---- depot (branche parametree) + dependances d'entrainement ------------------------------------------------------------
%cd /content
if not os.path.isdir('/content/LLM-and-Classifier'):
    !git clone -q -b {BRANCH} {REPO} /content/LLM-and-Classifier
%cd /content/LLM-and-Classifier
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only && git log --oneline -1
assert os.path.isdir('prophet_studio') and os.path.isfile('training/merge_lora.py'), 'mauvaise branche : prophet_studio/ ou training/merge_lora.py absent'
!pip install -q -e ".[train,serve]" huggingface_hub sentencepiece protobuf
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
# ---- llama.cpp du fork PrismML au tag du runtime de Studio : binaires CUDA (serveur, quantification) + source (conversion) --
import re, urllib.request
try:
    v = re.search(r'release (\d+)\.(\d+)', subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout)
    cu = (int(v[1]), int(v[2]))
except Exception:
    cu = (12, 4)
tag = '13.3' if cu >= (13, 3) else '12.8' if cu >= (12, 8) else '12.4'
BIN = os.path.abspath('bin/cuda'); os.makedirs(BIN, exist_ok=True)
if not os.path.exists(f'{BIN}/llama-server'):
    asset = f'llama-{LLAMA_TAG}-bin-linux-cuda-{tag}-x64.tar.gz'
    urllib.request.urlretrieve(f'https://github.com/PrismML-Eng/llama.cpp/releases/download/{LLAMA_TAG}/{asset}', '/content/llama.tgz')
    !tar -xzf /content/llama.tgz -C {BIN} --strip-components=1 2>/dev/null || tar -xzf /content/llama.tgz -C {BIN}
os.environ['LD_LIBRARY_PATH'] = BIN + ':' + os.environ.get('LD_LIBRARY_PATH', '')
if not os.path.isdir('/content/llama.cpp-src'):
    !git clone -q --depth 1 -b {LLAMA_TAG} https://github.com/PrismML-Eng/llama.cpp /content/llama.cpp-src || git clone -q --depth 1 -b prism https://github.com/PrismML-Eng/llama.cpp /content/llama.cpp-src
!ls {BIN} | grep -E 'llama-(server|quantize)$' ; ls /content/llama.cpp-src/convert_hf_to_gguf.py

Si les binaires precompiles refusent de demarrer (pilote CUDA trop ancien), compiler depuis la source (~10 min) :
```
!cd /content/llama.cpp-src && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=80 -DCMAKE_BUILD_TYPE=Release >/dev/null && cmake --build build -j8 --target llama-server llama-quantize >/dev/null && cp build/bin/* {BIN}/
```

## Enseignant (optionnel) : Bonsai 2 27B

Avec `USE_TEACHER = True`, Bonsai 2 27B (PQ2_0 : plus rapide sur Ampere) tourne en arriere-plan sur l'A100 et etiquette
l'entrainement : distributions `teacher_probs` (terme KL) et remplacement des etiquettes estimees (`needs_reasoning`).
Les etiquettes des regles font foi ; `teacher_disagrees` liste les desaccords a relire. Sans enseignant : regles seules.

In [ ]:
bonsai = None
if USE_TEACHER:
    # profil A100 sans clone niveau 0 ni vision : seuls les poids de Bonsai 2 27B PQ2_0 sont telecharges
    prof = open('scripts/profiles/a100-80gb.env').read().replace('JEV_MODE=dual', 'JEV_MODE=mono').replace('BONSAI_MMPROJ=gpu', 'BONSAI_MMPROJ=off')
    open('/content/teacher.env', 'w').write(prof)
    !PROFILE=/content/teacher.env SKIP_VENV=1 sh scripts/setup.sh
    bonsai = subprocess.Popen(['sh', 'scripts/start_bonsai.sh'], env=dict(os.environ, PROFILE='/content/teacher.env'),
                              stdout=open('/content/bonsai.log', 'w'), stderr=subprocess.STDOUT)
    import requests
    for _ in range(300):
        time.sleep(2)
        try:
            if requests.get('http://127.0.0.1:8080/health', timeout=2).json().get('status') == 'ok': break
        except Exception: pass
    print(requests.get('http://127.0.0.1:8080/health').json())

## Donnees

1. `training/make_synthetic_prophet.py` : exemples au format exact de Prophet (regles ; validation = gabarits jamais vus ;
   `prophet_calib.jsonl` = pre-tour tenu a l'ecart pour la calibration).
2. **Garde-fou** : actions benignes et ~40 % d'exemples defensifs des classes risquees (integres au generateur).
   Vos propres exemples (refus dans Studio, regles internes) s'ajoutent via `DRIVE/guard_risky.jsonl` (optionnel).
3. Trajectoires du computer use de Studio (`DRIVE/trajectories*.jsonl`, dossier de donnees de Studio `runs/`) -> DAgger.

In [ ]:
os.makedirs('data', exist_ok=True)
extra = f'{DRIVE}/{GUARD_EXTRA}'
args = f'--out data/prophet_train.jsonl --val data/prophet_val.jsonl --calib data/prophet_calib.jsonl --n {N_EXAMPLES}'
if os.path.exists(extra):
    args += f' --guard-extra {extra}'
else:
    print(f'pas de {extra} : famille guard retiree (voir la cellule precedente)')
if USE_TEACHER:
    args += f' --teacher http://127.0.0.1:8080 --workers {TEACHER_WORKERS} --think-if-below 0.85'
!python training/make_synthetic_prophet.py {args}
TRAJS = ' '.join(f'{DRIVE}/{f}' for f in sorted(os.listdir(DRIVE)) if f.startswith(('trajectories', 'desktop_trajectories')) and f.endswith('.jsonl'))
TRAIN_PARTS, VAL_PARTS = 'data/prophet_train.jsonl', 'data/prophet_val.jsonl'
if TRAJS:   # DAgger : pas escalades vers Bonsai dans le computer use de Studio
    !python training/make_from_trajectories.py --in {TRAJS} --out data/cu_train.jsonl --val data/cu_val.jsonl
    TRAIN_PARTS += ' data/cu_train.jsonl'; VAL_PARTS += ' data/cu_val.jsonl'
!cat {TRAIN_PARTS} > data/train.jsonl   # train_lora_rlcd.py melange lui-meme
!cat {VAL_PARTS} > data/val.jsonl
!wc -l data/train.jsonl data/val.jsonl data/prophet_calib.jsonl
!mkdir -p {DRIVE}/data && cp data/*.jsonl {DRIVE}/data/
if bonsai is not None:
    bonsai.terminate(); bonsai = None   # tout le GPU pour l'entrainement

## Entrainement RLCD-lite (QLoRA par defaut)

NLL sur les logits restreints aux etiquettes (+ KL vers Bonsai si enseignant), permutations d'options. Points de reprise
tous les 500 pas, copies dans Drive. `TRAIN_MODE = 'full'` : tous les poids (meilleur, plus long).

In [ ]:
mode = {'qlora': '--qlora', 'lora': '', 'full': '--full'}[TRAIN_MODE]
kl = 0.5 if USE_TEACHER else 0.0
!python training/train_lora_rlcd.py --model {MODEL} {mode} --data data/train.jsonl --val data/val.jsonl --out runs/{NAME} \
    --epochs 1 --bs 16 --accum 2 --max-len 1536 --loss nll --kl {kl} --permutations 2 --save-every 500 2>&1 | tail -40
!mkdir -p {DRIVE}/runs && cp -r runs/{NAME} {DRIVE}/runs/

## Fusion + GGUF

`training/merge_lora.py` recharge la base en bf16 (jamais en 4-bit), fusionne l'adaptateur, convertit en GGUF (script du
fork au tag du runtime de Studio) puis quantifie en Q8_0 et Q4_K_M. Outils verifies avant la fusion.

In [ ]:
src = f'--adapter runs/{NAME}' if TRAIN_MODE == 'qlora' else f'--merged runs/{NAME}/merged'
!python training/merge_lora.py {src} --llama-cpp /content/llama.cpp-src --quantize-bin {BIN}/llama-quantize \
    --gguf-dir runs --name {NAME} --quant {QUANTS}
!ls -la runs/*.gguf

## Calibration et mesure (sur le GGUF tel qu'il sera deploye)

Le clone est servi comme dans Studio (contexte 8 k par slot, KV q8_0) : `jev_clone.calibrate` ajuste temperature et seuils
**du pre-tour seulement** sur `prophet_calib.jsonl` (le garde-fou, la voix et les outils restent lus bruts), et
`training/eval_clone.py` mesure chaque famille (exactitude, ECE, confirmations inutiles, actions risquees manquees).

In [ ]:
GGUF = f'runs/{NAME}-{QUANTS.split(",")[0]}.gguf'
clone = subprocess.Popen([f'{BIN}/llama-server', '-m', GGUF, '--host', '127.0.0.1', '--port', '8081', '-ngl', '99', '-fa', 'on',
                          '-c', '32768', '-np', '4', '--cache-type-k', 'q8_0', '--cache-type-v', 'q8_0', '--reasoning-budget', '0',
                          '--no-mmproj'], stdout=open('/content/clone.log', 'w'), stderr=subprocess.STDOUT)
import requests
for _ in range(90):
    time.sleep(2)
    try:
        if requests.get('http://127.0.0.1:8081/health', timeout=2).json().get('status') == 'ok': break
    except Exception: pass
!python -m jev_clone.calibrate --server http://127.0.0.1:8081 --data data/prophet_calib.jsonl --out runs/calibration.json --target-precision 0.95
!python training/eval_clone.py --server http://127.0.0.1:8081 --data data/val.jsonl --json runs/eval_clone.json
clone.terminate()

In [ ]:
# ---- artefacts : Drive + telechargement direct -------------------------------------------------------------------------
!cp runs/{NAME}-*.gguf runs/{NAME}.manifest.json runs/calibration.json runs/eval_clone.json data/prophet_val.jsonl {DRIVE}/
!ls -la {DRIVE}
from google.colab import files
for f in [f'runs/{NAME}-{QUANTS.split(",")[0]}.gguf', 'runs/calibration.json']:
    files.download(f)   # ~0,9 Go pour 0.8B Q8_0 ; sinon recuperer depuis Google Drive

## Import dans Prophet Studio (machine Windows 11 + RTX 5060)

1. Copier `jev-clone-Q8_0.gguf` (Drive ou telechargement) sur la machine, par ex. `C:\Users\<vous>\Models\`.
2. Prophet Studio > **Modeles** > *Importer un GGUF* : chemin du fichier, role **Classifieur**, *Importer*. Studio le choisit
   comme classifieur (id `custom-s1-jev-clone-q8-0`) au prochain demarrage : redemarrer les modeles. Le S1 tourne sur CPU a
   cote de Bonsai 2 27B sur le GPU.
3. Quand il tourne : **Calibrer** (graines etiquetees livrees, lues sur ces poids-la : la calibration de reference).
   Variante : *Importer une calibration* -> `calibration.json` de ce notebook, pour ce classifieur.
   En ligne de commande : `python -m jev_clone.calibrate --server http://127.0.0.1:7881 --studio-model custom-s1-jev-clone-q8-0`.
4. Mesurer avant / apres sur votre machine (S1 de Studio sur le port 7881) :
   `python training/eval_clone.py --server http://127.0.0.1:7881 --data prophet_val.jsonl`
5. Iteration suivante : vos journaux de Studio (`runs/ledger.jsonl`, `runs/*trajectories.jsonl`) et vos refus du garde-fou
   dans Drive, puis relancer ce notebook.